In [1]:
import os
from dotenv import load_dotenv

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

load_dotenv("/home/jovyan/work/.env")
MONGO_URI = os.getenv("MONGO_URI")

spark = (
    SparkSession.builder
    .appName("FactibilidadPredictiva")
    .config("spark.mongodb.read.connection.uri", MONGO_URI)
    .config("spark.mongodb.write.connection.uri", MONGO_URI)
    .config(
        "spark.jars.packages",
        "org.mongodb.spark:mongo-spark-connector_2.12:10.1.1"
    )
    .getOrCreate()
)

In [2]:
df = (
    spark.read
    .format("mongodb")
    .option("database", "proyecto_bigdata")
    .option("collection", "Contenedor_Autos_Limpio1")
    .load()
)

print("Cantidad total registros:", df.count())

Cantidad total registros: 1685


In [4]:
variables_modelo = df.select(
    "precio",
    "kilometraje",
    "year",
    "marca",
    "modelo",
    "combustible",
    "ciudad"
)

variables_modelo.show(5, truncate=False)

+---------+-----------+----+-------+----------------------------+-----------+---------+
|precio   |kilometraje|year|marca  |modelo                      |combustible|ciudad   |
+---------+-----------+----+-------+----------------------------+-----------+---------+
|1.549E7  |44387.0    |2024|changan|Cs55 1.5t Elite 4x2 At 5p   |bencina    |araucania|
|8000000.0|75500.0    |2017|changan|Cs75                        |bencina    |santiago |
|7190000.0|92300.0    |2018|changan|Cx70 1.6                    |bencina    |araucania|
|6950000.0|87626.0    |2023|chery  |Arrizo 1.5                  |bencina    |santiago |
|1.248E7  |45050.0    |2021|chery  |Tiggo 1.5t Glx 4x2 Cvt At 5p|bencina    |santiago |
+---------+-----------+----+-------+----------------------------+-----------+---------+
only showing top 5 rows



In [5]:
print("Cantidad registros:", variables_modelo.count())

print("Marcas distintas:",
      variables_modelo.select("marca").distinct().count())

print("Combustibles distintos:",
      variables_modelo.select("combustible").distinct().count())

print("Ciudades distintas:",
      variables_modelo.select("ciudad").distinct().count())

Cantidad registros: 1685
Marcas distintas: 32
Combustibles distintos: 3
Ciudades distintas: 66


In [5]:
variables_modelo.printSchema()

root
 |-- precio: double (nullable = true)
 |-- kilometraje: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- marca: string (nullable = true)
 |-- modelo: string (nullable = true)
 |-- combustible: string (nullable = true)
 |-- ciudad: string (nullable = true)



In [6]:
total_original = 3627
total_final = 1685

porcentaje_conservado = (total_final / total_original) * 100
porcentaje_eliminado = 100 - porcentaje_conservado

print(f"Porcentaje conservado: {porcentaje_conservado:.2f}%")
print(f"Porcentaje eliminado: {porcentaje_eliminado:.2f}%")

Porcentaje conservado: 46.46%
Porcentaje eliminado: 53.54%


La construcción de un modelo predictivo para poder estimar el precio de un vehículo de manera coherente se realiza a través de diversas variables presentes en la base de datos.

Algunas de estas variables son kilometraje, año, modelo, combustible y ciudad. Estas variables entregan características relevantes del vehículo y del mercado comercial. Además, luego del proceso de validación y selección de datos, se conservaron 1.685 registros de un total inicial de 3.627 observaciones, equivalentes al 46.46% de los datos originales. Dentro de estos registros se identifican 65 marcas distintas, 4 tipos de combustible y 74 ciudades, lo que proporciona una cantidad suficiente de observaciones para el entrenamiento de modelos supervisados orientados a la predicción de precios.

Se evidencia una relación entre el precio, el año del vehículo y el kilometraje, lo que muestra señales de depreciación vehicular. En términos generales, los vehículos más nuevos tienden a presentar precios más altos, mientras que aquellos con un mayor kilometraje suelen disminuir su valor debido al desgaste asociado a su uso. Esto explica la utilización de estas variables para analizar y comprender el comportamiento del precio.

Por lo tanto, se puede concluir que existen las condiciones necesarias para desarrollar un modelo predictivo orientado a la estimación del precio de los vehículos usados, utilizando una base de datos con cobertura suficiente, diversidad de observaciones y variables relevantes para explicar el valor comercial de los vehículos.